In [ ]:
%load_ext autoreload
%autoreload 2
from machine_lib import * 
from mylib import *
# TSOps,
s = login()


In [ ]:
import pandas as pd
# 树表转换
# 读取 Excel 文件
file_path = "押品类目.xlsx"  # 替换为您的文件路径
data = pd.read_excel(file_path, header=None)

# 添加列名，假设 A 列为 "编号"，B 列为 "名称"
data.columns = ["编号", "名称"]

# 转换编号为字符串，避免整数处理导致的问题
data["编号"] = data["编号"].astype(str)

# 定义菜单生成逻辑
result = []

for _, row in data.iterrows():
    code = row["编号"].strip()  # 编号（移除可能的多余空格）
    name = row["名称"].strip()  # 名称（移除可能的多余空格）

    # 一级菜单
    if len(code) == 2:
        level_1 = name
        result.append([level_1, "", "", ""])

    # 二级菜单
    elif len(code) == 5:
        try:
            parent_code = code[:2]  # 前两位对应的一级编号
            level_1 = data.loc[data["编号"] == parent_code, "名称"].values[0]
            level_2 = name
            result.append([level_1, level_2, "", ""])
        except IndexError:
            print(f"Warning: 找不到一级菜单对应编号 {parent_code}")

    # 三级菜单
    elif len(code) == 8:
        try:
            parent_code = code[:2]  # 前两位对应的一级编号
            child_code = code[:5]  # 前五位对应的二级编号
            level_1 = data.loc[data["编号"] == parent_code, "名称"].values[0]
            level_2 = data.loc[data["编号"] == child_code, "名称"].values[0]
            level_3 = name
            result.append([level_1, level_2, level_3, ""])
        except IndexError:
            print(f"Warning: 找不到二级菜单或一级菜单对应编号 {child_code} 或 {parent_code}")

    # 四级菜单
    elif len(code) == 11:
        try:
            # print(1)
            # if code=='4008011010': 
            #     print(1)
            parent_code = code[:2]      # 前两位对应的一级编号
            child_code = code[:5]      # 前五位对应的二级编号
            sub_child_code = code[:8]  # 前十位对应的三级编号
            level_1 = data.loc[data["编号"] == parent_code, "名称"].values[0]
            level_2 = data.loc[data["编号"] == child_code, "名称"].values[0]
            level_3 = data.loc[data["编号"] == sub_child_code, "名称"].values[0]
            level_4 = name
            if level_4:  # 仅当四级菜单不为空时才加入
                result.append([level_1, level_2, level_3, level_4])
        except IndexError:
            print(f"Warning: 找不到三级菜单、二级菜单或一级菜单对应编号 {sub_child_code}、{child_code} 或 {parent_code}")
    else:
        print(code)

# 转换结果为 DataFrame
result_df = pd.DataFrame(result, columns=["一级菜单", "二级菜单", "三级菜单", "四级菜单"])

# 删除四级菜单为空的行
result_df = result_df[result_df["四级菜单"].notna() & (result_df["四级菜单"] != "")]

# 导出为 Excel 文件
result_df.to_excel("四级菜单转换结果_过滤空行.xlsx", index=False)

print("转换完成，结果已保存为四级菜单转换结果_过滤空行.xlsx")


In [ ]:
import openpyxl
# 拆分合并单元格
# 加载工作簿
wb = openpyxl.load_workbook('押品分类比对.xlsx')

# 选择要处理的工作表
sheet = wb.active  # 或者 wb['Sheet1']

# 将合并单元格范围复制到一个新的列表中
merged_ranges = list(sheet.merged_cells.ranges)

# 遍历合并单元格的所有范围
for merged_range in merged_ranges:
    # 获取合并单元格的范围的边界 (min_col, min_row, max_col, max_row)
    min_col, min_row, max_col, max_row = merged_range.bounds
    
    # 获取合并单元格左上角单元格的值
    start_cell = sheet.cell(row=min_row, column=min_col)
    value = start_cell.value  # 获取该坐标处单元格的值
    
    # 确保值是字符串类型，避免出现 'Cell' 对象的问题
    if value is not None:
        value = str(value)  # 转换为字符串类型
    
    # 拆分合并单元格
    sheet.unmerge_cells(str(merged_range))  # 拆分合并单元格
    
    # 填充拆分后的每个单元格的值
    for row in range(min_row, max_row + 1):
        for col in range(min_col, max_col + 1):
            cell = sheet.cell(row=row, column=col)
            cell.value = value  # 将值填充到每个单元格

# 保存修改后的工作簿
wb.save('your_file_modified.xlsx')


In [1]:
# 
import openpyxl
# 拆分合并单元格
# 加载工作簿
wb = openpyxl.load_workbook('押品分类比对.xlsx',data_only=True)
class Menu:
        def __init__(self,name,lv,op,old):
                self.name=name
                self.lv=lv
                self.op=op
                self.old=old
        def __str__(self):
              return self.name+"，"+str(self.lv)+","+str(self.op)+","+self.old
        def __hash__(self):   
              return hash(self.name) 
        def __eq__(self,other):   
              return self.name==other.name
# 选择要处理的工作表
sheet =  wb['新系统']
sheetwrite=wb['Sheet2']
list = [set() for _ in range(4)]

for row in sheet.iter_rows():
        for cell in row:
                if cell.column<5:
                    color=cell.fill.start_color
                    hyperlink= cell.hyperlink
                    rgb= color.rgb
                    print ((cell.value or "")+","+str(cell.column))
                    print(rgb)
                    changes=None
                    if hyperlink is not None:
                        parts = hyperlink.location.split('!')
                        if len(parts) == 2:
                            target_sheet_name = parts[0]
                            target_cell_address = parts[1]
                            target_sheet = wb[target_sheet_name]
                            target_cell = target_sheet[target_cell_address]
                            tvalue=target_cell.value
                            print(tvalue)
                            changes=tvalue.split(' 改为 ')
                            # print(changes[0])
                            # print(changes[1])
                            menu=Menu(changes[0],cell.column,1,changes[1]) 
                            list[cell.column-1].add(menu)
                    
                    # if rgb=='FFFFFF00':
                    #     menu=Menu(changes[0],cell.column,1,changes[1]) 
                    #     list[cell.column-1].add(menu)
                    icell=sheet.cell(row=cell.row,column=cell.column+10)  
                    if cell.row==14 and cell.column==3:
                        print(icell.value)
                    if icell.value=="原型有新系统没有":    
                        menu=Menu(cell.value,cell.column,2,'') 
                        list[cell.column-1].add(menu)
print("********************")
print(list)
for vset in list:
      print("=====")
      for m in vset:
            print(str(m))
for i in range(len(list)):
    codelen=2+(i)*3
    set=list[i]
    for menu in set:
        name=menu.name
        namecell=cell
        for row in sheetwrite.iter_rows(min_col=1,max_col=1):
            i
            for cell  in row:
                if cell.row!=1:    
                    if len(str(cell.value))==codelen:
                        tcell=sheetwrite.cell(row=cell.row,column=cell.column+1)   
                        if name== tcell.value: 
                             namecell=tcell
        column=namecell.column+5 if menu.op==1 else namecell.column+6
        sheetwrite.cell(row=namecell.row,column=column,value= menu.old  if menu.op==1 else 2)
wb.save("test1")        
                              
                              
            
# 将合并单元格范围复制到一个新的列表中

# 保存修改后的工作簿
# wb.save('your_file_modified.xlsx')


一级菜单,1
00000000
二级菜单,2
00000000
三级菜单,3
00000000
四级菜单,4
00000000
金融质押品,1
00000000
现金及其等价物,2
00000000
存单或存款,3
00000000
我行存单,4
00000000
金融质押品,1
00000000
现金及其等价物,2
00000000
存单或存款,3
00000000
他行存单,4
00000000
金融质押品,1
00000000
现金及其等价物,2
00000000
保证金,3
00000000
保证金,4
00000000
金融质押品,1
00000000
贵金属,2
FFFFFFFF
我行出售的贵金属,3
Values must be of type <class 'str'>
我行出售的贵金属,4
Values must be of type <class 'str'>
金融质押品,1
00000000
贵金属,2
00000000
金交所托管的贵金属,3
FFFFFFFF
黄金,4
FFFFFFFF
金融质押品,1
00000000
贵金属,2
00000000
金交所托管的贵金属,3
00000000
标准银、铂金等其他贵金属,4
FFFFFFFF
金融质押品,1
00000000
同业存单,2
Values must be of type <class 'str'>
同业存单,3
Values must be of type <class 'str'>
同业存单,4
Values must be of type <class 'str'>
金融质押品,1
00000000
债券,2
FFFFFFFF
国债,3
FFFFFFFF
储蓄国债,4
FFFFFFFF
金融质押品,1
00000000
债券,2
00000000
国债,3
00000000
凭证国债,4
FFFFFFFF
金融质押品,1
00000000
债券,2
00000000
国债,3
00000000
记账式国债,4
FFFFFFFF
金融质押品,1
00000000
债券,2
00000000
央票,3
FFFFFFFF
央行票据,4
FFFFFFFF
金融质押品,1
00000000
债券,2
00000000
地方政府债券,3
FFFFFF00
地方债 改为 地方政府债券
境内地

In [4]:
with open('11', 'r', encoding='utf - 8') as file:
        for line in file:
            s=line.split(' ')
            print(s[1])

getClrInfo

getClrInfoListById

getClrValueByClrIds

getListClr

getClrListByGuarantyNo

getClrInfoListByClrIds

saveContractAndGuaranty

getClrBalance

importGuarRelative

notvalidGuarRelative

getGuaRelativeListByNo

setClrSum

getGuarantySum

deleteGuarantyRelative、delGuarantyRelative

addGuarantyRelative

copyGuarRelative

getImportClrList

isAllowedReMortgInLine

getDependMortgageRateList

getClrRelativeRightCertList

findClrInfoByContractNos

findClrInfoByClrIds

getClrOwnerList

getClrAttachTable

getTaskDoneList

getTaskToDosList

getTaskEndList

getClrTaskWorking

getClrCount

getFinishedCount

saveCorpOrgInfo

deleteCorpOrg

updateStatus

initOrgBelong

saveOrg

delOrg
